# DATA 5310 Final Project — Crime Around the World (UNODC and World Bank)
**Homework 7: Data sources, cleaning, and visualization brainstorming**

**Group theme:** crime, at three scales, one per group member: Seattle, the United States, and the world. This notebook is the world component. It uses United Nations (UNODC) data on intentional homicide, on violent and sexual crime, and on property crime, by country, with World Bank indicators as the supporting source.

Only the international data are loaded here. Because the group's other two scales are Seattle and the United States, the notebook also compares the United States with eight other large high-income countries in section 6.

**Years analyzed: 2010–2024.** The group agreed that all three parts of the project use this window, so that the Seattle, United States and world data cover the same period and match each other (section 5).

**Questions this notebook is built to answer**
1. Is the homicide rate of the United States high or low compared with other large high-income countries?
2. Has the United States followed the same trend as those countries since 2010?
3. Do national income and inequality relate to homicide, and where does the United States sit?
4. Is the mix of property and violent crime in the United States unusual?

**Contents**
1. Load, sample, and describe the data (three UNODC files and one World Bank file)
2. Challenges and details to be aware of
3. Cleaning
4. Merging homicide with the violent-crime offenses (then property crime and World Bank indicators)
5. Coverage: which countries and years can we compare?
6. Exploratory look: the United States among its peer countries
7. Summary: what is worth investigating
8. Step 2: Visualization brainstorm
9. References

## 1. Load, sample, and describe the data

In [1]:
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Notebook lives in code/, data lives in data/ (also works from the repo root)
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
VIOLENT = DATA_DIR / "data_cts_violent_and_sexual_crime.xlsx"
HOMICIDE = DATA_DIR / "data_cts_intentional_homicide.xlsx"
ECON = DATA_DIR / "data_cts_corruption_and_economic_crime.xlsx"
WORLDBANK = DATA_DIR / "worldbank_indicators_2000_2024.csv"

# Each UNODC sheet has a 2-row banner (contact + download date); the real header is row 3.
viol_raw = pd.read_excel(VIOLENT, header=2)
hom_raw = pd.read_excel(HOMICIDE, sheet_name="data_cts_intentional_homicide", header=2)
econ_raw = pd.read_excel(ECON, header=2)
wb_raw = pd.read_csv(WORLDBANK, keep_default_na=False, na_values=[""])   # keep "NA" (Namibia) from being read as missing
print("violent & sexual crime:", viol_raw.shape)
print("intentional homicide  :", hom_raw.shape)
print("corruption & economic crime:", econ_raw.shape)
print("World Bank indicators:", wb_raw.shape)

violent & sexual crime: (34974, 13)
intentional homicide  : (126082, 13)
corruption & economic crime: (27079, 13)
World Bank indicators: (19950, 6)


### 1a. Source files

The three UNODC files come from the **United Nations Office on Drugs and Crime (UNODC)**, data portal <https://data.unodc.org>. National authorities report to UNODC through the annual Crime Trends Survey (CTS).

| File | Sheet | What it is |
|---|---|---|
| `data_cts_violent_and_sexual_crime.xlsx` | 1 sheet | Violent offenses by type (serious assault, robbery, rape, kidnapping, ...) plus victims of assault and sexual violence by relationship to the perpetrator. File dated 2026-09-28. Source page: <https://data.unodc.org/datareport/violent-offences> (UNODC, 2026a). |
| `data_cts_intentional_homicide.xlsx` | `data_cts_intentional_homicide` sheet | Country-reported homicide victims (and arrests, convictions, prison deaths), with breakdowns by mechanism, relationship to perpetrator, and situational context. File dated 2026-07-12. Source page: <https://data.unodc.org/datareport/hom-victim> (UNODC, 2026b). |
| `data_cts_corruption_and_economic_crime.xlsx` | 1 sheet | Offenses by type: burglary, theft, theft of a motorized vehicle, plus fraud, corruption, money laundering, cyber offenses and environmental offenses. File dated 2026-07-12; downloaded 2026-11-14 from <https://data.unodc.org/datareport/econ-corruption> (UNODC, 2026c). Added so property crime can be compared with violent crime. |
| `worldbank_indicators_2000_2024.csv` | 1 file | Three series (GDP per capita, the Gini index and population) from the World Bank's *World Development Indicators*, 2000–2024, downloaded 2026-11-14 from the World Bank API (<https://api.worldbank.org>) by `code/fetch_worldbank.py` (World Bank, 2026). Long format: `iso3`, `country`, `year`, `indicator_code`, `indicator`, `value`. Described in section 1e. |

Contact on the UNODC files: `unodc-stats@un.org`.

Full citations are listed under References at the end of the notebook.


### 1b. Violent and sexual crime file

In [2]:
viol_raw.sample(8, random_state=1)

,Iso3_code,Country,Region,Subregion,Indicator,Dimension,Category,Sex,Age,Year,Unit of measurement,VALUE,Source
18642,JOR,Jordan,Asia,Western Asia,Violent offences,by type of offence,Kidnapping,Total,Total,2021,Counts,162.000000,CTS
1288,ATG,Antigua and Barbuda,Americas,Latin America and the Caribbean,Violent offences,by type of offence,Sexual violence,Total,Total,2017,Counts,83.000000,CTS
6693,CHE,Switzerland,Europe,Western Europe,Violent offences,by type of offence,Sexual violence: Sexual assault,Total,Total,2018,"Rate per 100,000 population",7.808515,CTS
33602,URY,Uruguay,Americas,Latin America and the Caribbean,Violent offences,by type of offence,Sexual violence: Sexual assault,Total,Total,2019,Counts,1198.000000,CTS
21932,LVA,Latvia,Europe,Northern Europe,Victims of sexual violence,by relationship to perpetrator,Perpetrator unknown to the victim,Female,Total,2016,Counts,31.000000,CTS
25245,MNG,Mongolia,Asia,Eastern Asia,Victims of serious assault,by relationship to perpetrator,Intimate partner or family member,Female,Total,2015,"Rate per 100,000 population",0.932771,CTS
12497,FIN,Finland,Europe,Northern Europe,Violent offences,by type of offence,Robbery,Total,Total,2008,Counts,1696.000000,CTS
31047,SVK,Slovakia,Europe,Eastern Europe,Violent offences,by type of offence,Sexual violence: Other acts of sexual violence,Total,Total,2018,"Rate per 100,000 population",9.038748,CTS


**Columns:** `Iso3_code`, `Country`, `Region`, `Subregion` (jurisdiction and UN geographic grouping); `Indicator` (`Violent offences`, `Victims of serious assault`, `Victims of sexual violence`); `Dimension` and `Category` (offense type, or victim–perpetrator relationship); `Sex`; `Age` (always `Total` in this file); `Year`; `Unit of measurement` (`Counts` or `Rate per 100,000 population`); `VALUE`; `Source`.

One row = jurisdiction × indicator × category × sex × year × unit. The same measurement appears twice, once as a count and once as a rate.

In [3]:
display(pd.Series({"rows": len(viol_raw), "jurisdictions": viol_raw["Country"].nunique(),
                   "years": f"{viol_raw['Year'].min()}-{viol_raw['Year'].max()}",
                   "missing VALUE": viol_raw["VALUE"].isna().sum()}).to_frame("violent file"))
display(viol_raw.groupby(["Indicator", "Dimension"]).agg(rows=("VALUE", "size"), categories=("Category", "nunique")))
print(sorted(viol_raw.loc[viol_raw.Indicator == "Violent offences", "Category"].unique()))

,violent file
rows,34974
jurisdictions,159
years,2003-2024
missing VALUE,0


,,rows,categories
Indicator,Dimension,,
Victims of serious assault,by relationship to perpetrator,4268,4
Victims of sexual violence,by relationship to perpetrator,4248,4
Violent offences,by type of offence,26458,10


['Acts intended to induce fear or emotional distress', 'Acts intended to induce fear or emotional distress: Cyber-related', 'Kidnapping', 'Robbery', 'Serious assault', 'Sexual Exploitation', 'Sexual violence', 'Sexual violence: Other acts of sexual violence', 'Sexual violence: Rape', 'Sexual violence: Sexual assault']


### 1c. Intentional homicide file

In [4]:
hom_raw.sample(8, random_state=1)

,Iso3_code,Country,Region,Subregion,Indicator,Dimension,Category,Sex,Age,Year,Unit of measurement,VALUE,Source
100277,VCT,Saint Vincent and the Grenadines,Americas,Latin America and the Caribbean,Victims of intentional homicide,by mechanisms,Another weapon - sharp object,Total,Total,2010,"Rate per 100,000 population",10.003774,CTS
108261,MNG,Mongolia,Asia,Eastern Asia,Victims of intentional homicide,Total,Total,Male,30-44,2007,"Rate per 100,000 population",37.829603,CTS
119752,GBR_NI,United Kingdom (Northern Ireland),Europe,Northern Europe,Persons arrested/suspected for intentional hom...,Total,Total,Male,30-44,2019,"Rate per 100,000 population",5.560498,CTS
11330,NZL,New Zealand,Oceania,Australia and New Zealand,Victims of intentional homicide,by relationship to perpetrator,Intimate partner or family member: Intimate pa...,Total,Total,2019,Counts,14.000000,CTS/NP
78673,ARG,Argentina,Americas,Latin America and the Caribbean,Victims of intentional homicide,by relationship to perpetrator,Intimate partner or family member: Intimate pa...,Male,Total,2019,"Rate per 100,000 population",0.233103,CTS
93492,RUS,Russian Federation,Europe,Eastern Europe,Victims of intentional homicide,by situational context,Interpersonal homicide,Total,Total,2017,"Rate per 100,000 population",1.065076,GSH 2023 Revision/CTS
92502,GUY,Guyana,Americas,Latin America and the Caribbean,Victims of intentional homicide,by situational context,Other criminal activities,Male,Total,2021,"Rate per 100,000 population",21.878591,CTS
55818,FRA,France,Europe,Western Europe,Persons arrested/suspected for intentional hom...,Total,Total,Female,0-9,2022,Counts,0.000000,CTS


In [5]:
display(pd.Series({"rows": len(hom_raw), "jurisdictions": hom_raw["Country"].nunique(),
                   "years": f"{hom_raw['Year'].min()}-{hom_raw['Year'].max()}",
                   "missing VALUE": hom_raw["VALUE"].isna().sum()}).to_frame("homicide file"))
display(hom_raw.groupby(["Indicator", "Dimension"]).agg(rows=("VALUE", "size"), categories=("Category", "nunique")))
for dim in ["by mechanisms", "by relationship to perpetrator", "by situational context"]:
    print(dim, "->", list(hom_raw.loc[(hom_raw.Indicator == "Victims of intentional homicide") & (hom_raw.Dimension == dim), "Category"].unique()))

,homicide file
rows,126082
jurisdictions,208
years,2003-2024
missing VALUE,0


rows  categories
Indicator                                          Dimension                                        
Death due to intentional homicide in prison        Total                            1628           1
Persons arrested/suspected for intentional homi... Total                           20914           1
                                                   by citizenship                   1859           2
Persons convicted for intentional homicide         Total                            2881           1
Victims of intentional homicide                    Total                           43060           1
                                                   by citizenship                   2031           2
                                                   by mechanisms                   10539           6
                                                   by relationship to perpetrator  26700           6
                                                   by situational context          16470           6

by mechanisms -> ['Firearms or explosives', 'Firearms or explosives - firearms', 'Another weapon', 'Another weapon - sharp object', 'Without a weapon/ other Mechanism', 'Unspecified means']
by relationship to perpetrator -> ['Intimate partner or family member', 'Intimate partner or family member: Intimate partner', 'Intimate partner or family member: Family member', 'Other Perpetrator known to the victim', 'Perpetrator unknown to the victim', 'Perpetrator to victim relationship unknown']
by situational context -> ['Organized criminal groups or gangs', 'Other criminal activities', 'Interpersonal homicide', 'Socio-political homicide', 'Socio-political homicide - terrorist offences', 'Unknown types of homicide']


Same structure as the violent file (same 13 columns), so the two can be joined on `Iso3_code`/`Country` and `Year`. For this project the key indicator is Victims of intentional homicide, `Dimension == "Total"`, `Sex == "Total"`: one row = country × year × unit. The other dimensions parse the same victims by how (e.g. firearm), by whom (relationship) and why (situational context).

### 1d. Corruption and economic crime file (property crime)

In [6]:
econ_raw.sample(8, random_state=1)

,Iso3_code,Country,Region,Subregion,Indicator,Dimension,Category,Sex,Age,Year,Unit of measurement,VALUE,Source
6069,ECU,Ecuador,Americas,Latin America and the Caribbean,Offences,by type of offence,Theft,Total,Total,2014,Counts,24300.000000,CTS
16322,DNK,Denmark,Europe,Northern Europe,Offences,by type of offence,Smuggling of migrants,Total,Total,2020,"Rate per 100,000 population",1.114630,CTS
24350,CZE,Czechia,Europe,Eastern Europe,Offences,by type of offence,Unlawful access to a computer system,Total,Total,2014,"Rate per 100,000 population",6.360993,CTS
23771,LTU,Lithuania,Europe,Northern Europe,Offences,by type of offence,Money laundering,Total,Total,2015,"Rate per 100,000 population",3.372607,CTS
3142,ECU,Ecuador,Americas,Latin America and the Caribbean,Offences,by type of offence,Burglary,Total,Total,2003,Counts,10224.000000,CTS
19781,BEL,Belgium,Europe,Western Europe,Offences,by type of offence,Theft,Total,Total,2016,"Rate per 100,000 population",2050.588031,CTS
5226,ISR,Israel,Asia,Western Asia,Offences,by type of offence,Theft,Total,Total,2006,Counts,114980.000000,CTS
22770,MAC,"China, Macao Special Administrative Region",Asia,Eastern Asia,Offences,by type of offence,Fraud,Total,Total,2018,"Rate per 100,000 population",181.398116,CTS


In [7]:
display(pd.Series({"rows": len(econ_raw), "jurisdictions": econ_raw["Country"].nunique(),
                   "years": f"{econ_raw['Year'].min()}-{econ_raw['Year'].max()}",
                   "missing VALUE": econ_raw["VALUE"].isna().sum()}).to_frame("economic crime file"))
display(econ_raw.groupby(["Indicator", "Category"]).agg(rows=("VALUE", "size"), jurisdictions=("Country", "nunique")))

,economic crime file
rows,27079
jurisdictions,157
years,2003-2024
missing VALUE,0


rows  jurisdictions
Indicator                    Category                                                               
Acts against the environment Acts involving the movement of dumping of waste      748             69
                             Acts that cause environmental pollution or degr...   906             77
                             Acts that result in the depletion of degradatio...   786             68
                             Trade or possession of protected or prohibited ...   710             64
Offences                     Burglary                                            3598            138
                             Corruption                                          1597            108
                             Corruption: Bribery                                 1652            107
                             Corruption: Other acts of corruption                1425             95
                             Fraud                                               1557            100
                             Fraud: Cyber-related (Cy)                            909             70
                             Money laundering                                    1320             92
                             Smuggling of migrants                               1579             98
                             Theft                                               4124            146
                             Theft: of a motorized vehicle                       3854            141
                             Unlawful access to a computer system                 920             83
                             Unlawful interception or access of computer data     594             60
                             Unlawful interference with a computer system or...   800             72

Same 13 columns and the same one row = jurisdiction × offense category × year × unit structure as the other two files (`Sex` and `Age` are always `Total`). At this point of the project only three categories matter: Burglary, Theft and Theft: of a motorized vehicle. These are the best definitions of property crime that the UN data sources has.

### 1e. World Bank indicators file

In [8]:
wb_raw.sample(8, random_state=1)

,iso3,country,year,indicator_code,indicator,value
13646,EUU,European Union,2003,SP.POP.TOTL,population,4.325129e+08
7632,PST,Post-demographic dividend,2017,SI.POV.GINI,gini,NaN
15935,DOM,Dominican Republic,2014,SP.POP.TOTL,population,1.031553e+07
8181,AZE,Azerbaijan,2018,SI.POV.GINI,gini,NaN
6160,TUN,Tunisia,2014,NY.GDP.PCAP.PP.CD,gdp_pc_ppp,1.094689e+04
15245,BRN,Brunei Darussalam,2004,SP.POP.TOTL,population,3.529110e+05
19727,UZB,Uzbekistan,2022,SP.POP.TOTL,population,3.493896e+07
13957,LDC,Least developed countries: UN classification,2017,SP.POP.TOTL,population,1.008261e+09


In [9]:
wb_is_country = wb_raw["iso3"].isin(set(hom_raw["Iso3_code"]) | set(viol_raw["Iso3_code"]))     # rows that match a UNODC country
display(pd.Series({"rows": len(wb_raw), "countries and aggregates": wb_raw["country"].nunique(),
                   "of which match a UNODC country": wb_raw.loc[wb_is_country, "iso3"].nunique(),
                   "years": f"{wb_raw['year'].min()}-{wb_raw['year'].max()}", "missing value": wb_raw["value"].isna().sum()}).to_frame("World Bank file"))
(wb_raw[wb_is_country].groupby(["indicator", "indicator_code"], sort=False)["value"]
    .agg(rows="size", share_filled=lambda x: round(x.notna().mean(), 2)).reset_index())

,World Bank file
rows,19950
countries and aggregates,266
of which match a UNODC country,196
years,2000-2024
missing value,5318


,indicator,indicator_code,rows,share_filled
0,gdp_pc_ppp,NY.GDP.PCAP.PP.CD,4900,0.92
1,gini,SI.POV.GINI,4900,0.37
2,population,SP.POP.TOTL,4900,1.00


**Columns:** `iso3` and `country` (the country, or a World Bank aggregate such as a region or income group), `year`, `indicator_code` (the World Bank's own code), `indicator` (a short name we gave that code) and `value`.

One row = country (or aggregate) × year × indicator. Every country has a row for every year and indicator, so years the World Bank has no figure for appear as rows with a missing `value` rather than being absent. Coverage therefore varies a lot by indicator (table above): population is complete, the Gini index exists for only about a third of country-years.

The three UNODC files contain crime figures and nothing else: no income, inequality or population. The Seattle data and the US data will likely have these characteristics, so this data file is an attempt to satisfy that requirement.
1. Research question 3 asks how income and inequality relate to homicide.
2. Describing the peer countries (section 6): income and inequality show how alike the United States and its peers are. Population is used to check the population implied by UNODC's counts and rates (section 4b).

This is the "core data set plus supporting source" arrangement the assignment describes: UNODC is the core, the World Bank file is the one supporting source, and they join on the ISO country code and year.

`code/fetch_worldbank.py`. The World Bank does not offer these three series as a ready-made files to download. Its bulk download is the entire indicator database (about 1,500 series), and its website tool exports only what you select by hand. The script is the repeatable version of that hand selection: it asks the World Bank's public API for each of the three indicator codes (all countries, 2000–2024) and stacks the answers into one CSV. It does not filter, fill or recalculate anything, so every value is exactly as the World Bank publishes it; the only thing added is the short `indicator` name.

Some issues to be aware of: aggregates (World, regions, income groups) are in the file and do not match any UNODC country in the merge; UNODC's three UK nations each receive the UK-wide value; a few territories UNODC reports have no World Bank entry (section 4b).

## 2. Challenges and details to be aware of
1. **Counts and rates are stacked in one `VALUE` column** in both files. Any summary must filter `Unit of measurement` first. We compare countries using rates per 100,000.
2. **Overlapping categories.** Parent/child pairs (`Sexual violence` vs `Sexual violence: Rape`; homicide `Firearms or explosives` vs `... - firearms`; `Intimate partner or family member` vs `: Intimate partner`) would double count if summed. We use top-level categories for totals and shares.
3. **Homicide is far better measured than other violence.** It is defined and recorded fairly consistently and is hard to hide. Assault, rape and robbery depend heavily on legal definitions and on whether victims report to the police, so a high recorded rate might mean better reporting as opposed to more violence.
4. **Uneven coverage.** Homicide rates are available for roughly 120–165 countries a year; the violent-offense categories for far fewer, with some starting only in 2010–2014. 2023–2024 are partial in both files.
5. **Age groups in the homicide file.** Unlike the violent file, many homicide rows are split by age group (0-9, 10-14, ..., 60+), and rate rows are sparser than count rows. Always filter `Age == "Total"` (and the right `Dimension`/`Sex`), otherwise totals are double counted. Missing victim rates can sometimes be recovered from counts if a population is known (tried in section 3).
6. **Jurisdictions are not always countries.** The UK is split into England & Wales, Scotland and Northern Ireland; Iraq has "Iraq", "Central Iraq" and "Kurdistan Region"; Hong Kong, Macao, Puerto Rico, Kosovo, Palestine, the Holy See also appear. Overlapping entries (like Iraq) can duplicate a country in a merge.
7. **Small populations give volatile rates** (small island states, Liechtenstein, Monaco): a handful of events can double a rate.
8. **Mixed `Source` values** (`CTS`, `CTS/NGO`, `CTS/MoI`, `GSH 2023 Revision/CTS`, ...) — some figures are UNODC revisions rather than direct country reports.
9. **Categories will need mapping across the three scales.** UNODC homicide is directly comparable with city and national homicide counts, but assault, robbery, rape and theft definitions will have to be matched by hand when the group combines its data.
10. **Property crime is recorded even less consistently than violent crime.** Whether shoplifting, bicycle theft or attempted offenses count as `Theft`, and whether vehicle theft is included in it, differs by country, and most property crime is never reported to the police.

In [10]:
print("Violent file, share of rows by Source:")
print((viol_raw["Source"].value_counts(normalize=True).head(4) * 100).round(1).to_string())
print("\nHomicide file, share of rows by Source:")
print((hom_raw["Source"].value_counts(normalize=True).head(6) * 100).round(1).to_string())

key = ["Iso3_code", "Country", "Indicator", "Dimension", "Category", "Sex", "Age", "Year", "Unit of measurement"]
print("\nDuplicate keys (incl. Age) - violent:", viol_raw.duplicated(key).sum(), "| homicide:", hom_raw.duplicated(key).sum())
print("Zero values - violent: %.1f%% | homicide: %.1f%%" % (100 * (viol_raw.VALUE == 0).mean(), 100 * (hom_raw.VALUE == 0).mean()))
print("\nISO codes shared by more than one jurisdiction name (homicide file):")
multi = hom_raw.groupby("Iso3_code")["Country"].nunique()
print(hom_raw[hom_raw["Iso3_code"].isin(multi[multi > 1].index)].groupby("Iso3_code")["Country"].unique().to_string())

Violent file, share of rows by Source:
Source
CTS        94.9
CTS/NGO     1.5
CTS/MoI     1.3
CTS/PPR     0.5

Homicide file, share of rows by Source:
Source
CTS                      72.9
GSH 2023 Revision/CTS     4.9
CTS/GSH 2023 Revision     2.4
CTS/SDG                   1.7
CTS/NGO                   1.5
CTS/MoI                   1.3

Duplicate keys (incl. Age) - violent: 0 | homicide: 0
Zero values - violent: 5.0% | homicide: 20.5%

ISO codes shared by more than one jurisdiction name (homicide file):
Series([], )


## 3. Cleaning
1. Tidy column names, convert `Unit of measurement` to `count`/`rate`, keep `age` (constant `Total` in the violent file, but the homicide file also has age-group rows that must be filtered out to avoid double counting).
2. Build the country-year homicide table (victims, total sex, rate and count).
3. Try to recover missing homicide rates from counts when a population is known for that country-year.
4. Build a country-year violent-offense table with short offense labels (top-level categories only).
5. Build a country-year property-offense table (burglary, theft, vehicle theft) from the third file.

In [11]:
def tidy(df):
    out = df.rename(columns={"Iso3_code": "iso3", "Country": "country", "Region": "region", "Subregion": "subregion",
                             "Indicator": "indicator", "Dimension": "dimension", "Category": "category", "Sex": "sex",
                             "Age": "age", "Year": "year", "Unit of measurement": "unit", "VALUE": "value", "Source": "source"})
    out["unit"] = out["unit"].map({"Counts": "count", "Rate per 100,000 population": "rate"})
    out["country"] = out["country"].str.strip()
    return out

viol, hom = tidy(viol_raw), tidy(hom_raw)

# --- homicide victims, totals: one row per country-year, rate & count side by side
hv = hom[(hom.indicator == "Victims of intentional homicide") & (hom.dimension == "Total") & (hom.sex == "Total") & (hom.age == "Total")]
hom_cy = (hv.pivot_table(index=["iso3", "country", "region", "subregion", "year"], columns="unit", values="value", aggfunc="first")
            .reset_index().rename(columns={"count": "hom_count", "rate": "hom_rate"}))
hom_cy.columns.name = None
print("country-years:", len(hom_cy), "| with rate:", hom_cy.hom_rate.notna().sum(), "| count only:", (hom_cy.hom_rate.isna() & hom_cy.hom_count.notna()).sum())
hom_cy.head()

country-years: 3012 | with rate: 3005 | count only: 7


,iso3,country,region,subregion,year,hom_count,hom_rate
0,ABW,Aruba,Americas,Latin America and the Caribbean,2003,4.0,4.333225
1,ABW,Aruba,Americas,Latin America and the Caribbean,2004,2.0,2.145635
2,ABW,Aruba,Americas,Latin America and the Caribbean,2005,6.0,6.367399
3,ABW,Aruba,Americas,Latin America and the Caribbean,2006,5.0,5.241502
4,ABW,Aruba,Americas,Latin America and the Caribbean,2007,3.0,3.106008


In [12]:
# Recover missing rates: infer population (count / rate * 100,000) from ANY homicide row of the same
# country-year that has both a count and a positive rate (e.g. arrests, other victim breakdowns).
both = (hom[hom.age == "Total"].pivot_table(index=["country", "year"], columns="unit", values="value", aggfunc="first")
          .dropna().query("rate > 0 and count > 0"))
pop_cy = (both["count"] / both["rate"] * 1e5).groupby(level=[0, 1]).median().rename("pop").reset_index()
hom_cy = hom_cy.merge(pop_cy, on=["country", "year"], how="left")

fill = hom_cy["hom_rate"].isna() & hom_cy["hom_count"].notna() & hom_cy["pop"].notna()
hom_cy.loc[fill, "hom_rate"] = hom_cy.loc[fill, "hom_count"] / hom_cy.loc[fill, "pop"] * 1e5
hom_cy["rate_recovered"] = fill
print("rates recovered from counts + inferred population:", fill.sum(), "(of", (hom_cy.hom_rate.isna() & hom_cy.hom_count.notna()).sum(), "count-only rows)")
print("country-years with a homicide rate now:", hom_cy.hom_rate.notna().sum())

# sanity: recovered rates should look like the reported ones
print(hom_cy.groupby("rate_recovered")["hom_rate"].describe()[["count", "50%", "mean", "max"]].round(2))
hom_cy = hom_cy.dropna(subset=["hom_rate"])

rates recovered from counts + inferred population: 0 (of 7 count-only rows)
country-years with a homicide rate now: 3005
                 count   50%  mean     max
rate_recovered                            
False           3005.0  2.74  7.98  107.64


In [13]:
# --- violent offenses: top-level categories, short labels
short = {"Serious assault": "assault", "Robbery": "robbery", "Kidnapping": "kidnapping", "Sexual violence: Rape": "rape",
         "Sexual violence": "sexual_violence_all"}
vo = viol[(viol.indicator == "Violent offences") & (viol.unit == "rate") & viol.category.isin(short)].copy()
vo["offense"] = vo["category"].map(short)
viol_cy = vo.pivot_table(index=["iso3", "country", "year"], columns="offense", values="value", aggfunc="first").reset_index()
viol_cy.columns.name = None
viol_cy = viol_cy.rename(columns={c: f"{c}_rate" for c in short.values()})
viol_geo = viol.drop_duplicates("country").set_index("country")[["region", "subregion"]]   # region labels from the violent file itself
viol_cy["region"] = viol_cy["country"].map(viol_geo["region"]); viol_cy["subregion"] = viol_cy["country"].map(viol_geo["subregion"])
print(viol_cy.shape)
viol_cy.head()

(2351, 10)


,iso3,country,year,assault_rate,kidnapping_rate,rape_rate,robbery_rate,sexual_violence_all_rate,region,subregion
0,ALB,Albania,2005,5.363832,NaN,1.592896,6.469106,2.893218,Europe,Southern Europe
1,ALB,Albania,2006,8.063436,0.426117,1.311128,5.375624,2.851703,Europe,Southern Europe
2,ALB,Albania,2007,7.244730,0.430052,1.356319,4.068958,3.341177,Europe,Southern Europe
3,ALB,Albania,2008,12.128581,0.334121,1.303071,4.811338,2.539317,Europe,Southern Europe
4,ALB,Albania,2009,12.797657,0.168835,1.148075,5.706607,2.802653,Europe,Southern Europe


In [14]:
# --- property offenses (third UNODC file): burglary, theft, and vehicle theft (a child of theft: never add it to theft)
econ = tidy(econ_raw)
pshort = {"Burglary": "burglary", "Theft": "theft", "Theft: of a motorized vehicle": "vehicle_theft"}
po = econ[(econ.indicator == "Offences") & (econ.unit == "rate") & econ.category.isin(pshort)].copy()
po["offense"] = po["category"].map(pshort)
prop_cy = po.pivot_table(index=["iso3", "country", "year"], columns="offense", values="value", aggfunc="first").reset_index()
prop_cy.columns.name = None
prop_cy = prop_cy.rename(columns={c: f"{c}_rate" for c in pshort.values()})
prop_cy["property_rate"] = prop_cy["burglary_rate"] + prop_cy["theft_rate"]      # needs both; vehicle theft sits inside theft
print(prop_cy.shape, "| countries:", prop_cy.country.nunique(), "| country-years with burglary + theft:", prop_cy.property_rate.notna().sum())
prop_cy.head()

(2133, 7) | countries: 151 | country-years with burglary + theft: 1750


,iso3,country,year,burglary_rate,theft_rate,vehicle_theft_rate,property_rate
0,ALB,Albania,2005,6.469106,97.069105,23.015715,103.538211
1,ALB,Albania,2006,5.375624,95.679555,17.405222,101.055179
2,ALB,Albania,2007,4.068958,98.581257,14.456379,102.650215
3,ALB,Albania,2008,4.811338,106.851796,13.298003,111.663134
4,ALB,Albania,2009,5.706607,103.765702,17.018521,109.472309


In [15]:
# Flag entries that are not whole sovereign states, so they can be excluded when comparing countries
non_country = ["United Kingdom (England and Wales)", "United Kingdom (Northern Ireland)", "United Kingdom (Scotland)",
               "China, Hong Kong Special Administrative Region", "China, Macao Special Administrative Region", "Puerto Rico",
               "Kosovo under UNSCR 1244", "Holy See", "State of Palestine", "Iraq (Central Iraq)", "Iraq (Kurdistan Region)"]
hom_cy["territory_or_subnational"] = hom_cy["country"].isin(non_country)
viol_cy["territory_or_subnational"] = viol_cy["country"].isin(non_country)
hom_cy["small_pop"] = hom_cy["pop"] < 1e6
print("flagged non-country entries (homicide):", hom_cy.loc[hom_cy.territory_or_subnational, "country"].nunique())
print("small-population (<1M) country-years:", int(hom_cy.small_pop.sum()), "of", len(hom_cy))

flagged non-country entries (homicide): 11
small-population (<1M) country-years: 654 of 3005


## 4. Merging homicide with the violent offenses
Goal: one country-year panel with a homicide rate and the other violent-offense rates side by side, so we can ask whether they move together and where they disagree. Key: `iso3` + `country` + `year` (country name is included because some ISO codes cover several jurisdictions, e.g. Iraq). An outer join keeps every country-year that appears in either file.

In [16]:
keys = ["iso3", "country", "year"]
assert not hom_cy.duplicated(keys).any() and not viol_cy.duplicated(keys).any()

panel_all = (hom_cy[keys + ["region", "subregion", "hom_rate", "hom_count", "rate_recovered", "pop", "small_pop", "territory_or_subnational"]]
             .merge(viol_cy.drop(columns=["territory_or_subnational", "region", "subregion"]), on=keys, how="outer", indicator=True))
print(panel_all["_merge"].value_counts().rename({"both": "in both files", "left_only": "homicide only", "right_only": "violent-offenses only"}))

# violent-only rows lack region labels from the homicide side: take them from the violent file
for c in ["region", "subregion"]:
    panel_all[c] = panel_all[c].fillna(panel_all["country"].map(viol_geo[c]))
print("rows still missing a region:", panel_all["region"].isna().sum())
panel_all.loc[panel_all["_merge"] == "right_only", "country"].unique()

_merge
in both files            2024
homicide only             981
violent-offenses only     327
Name: count, dtype: int64
rows still missing a region: 0


<ArrowStringArray>
[                           'Andorra',               'United Arab Emirates',                            'Armenia',                'Antigua and Barbuda',                         'Azerbaijan',
                            'Burundi',                            'Belgium',                              'Benin',                            'Bahrain',             'Bosnia and Herzegovina',
                            'Belarus',   'Bolivia (Plurinational State of)',                             'Bhutan',                           'Botswana',                        'Switzerland',
                              'Chile',                              'China',                      'Côte d’Ivoire',                           'Cameroon',                         'Cabo Verde',
                             'Cyprus',                            'Czechia',                            'Denmark',                 'Dominican Republic',                            'Algeria',
                          

In [17]:
panel_all = panel_all.drop(columns="_merge")
# flags were only defined on homicide rows; recompute them for every row of the merged panel
panel_all["territory_or_subnational"] = panel_all["country"].isin(non_country)
panel_all["small_pop"] = panel_all.groupby("country")["pop"].transform("median") < 1e6
print("panel:", panel_all.shape, "| countries:", panel_all.country.nunique(), "| years:", panel_all.year.min(), "-", panel_all.year.max())
rate_cols = ["hom_rate", "assault_rate", "robbery_rate", "rape_rate", "kidnapping_rate"]
panel_all.groupby("year")[rate_cols].count().tail(10)

panel: (3332, 16) | countries: 210 | years: 2003 - 2024


,hom_rate,assault_rate,robbery_rate,rape_rate,kidnapping_rate
year,,,,,
2015,134,106,95,103,98
2016,133,103,94,101,94
2017,133,108,97,106,100
2018,133,108,88,109,97
2019,140,105,86,105,93
2020,136,101,79,102,89
2021,126,103,92,102,95
2022,119,96,84,94,87
2023,108,93,84,90,84


### 4a. Adding property crime
A left join onto the panel (same key), so property rates are attached to country-years that already have homicide or violent-offense data; property-only country-years are counted but not added.

In [18]:
prop_cols = ["burglary_rate", "theft_rate", "vehicle_theft_rate", "property_rate"]
assert not prop_cy.duplicated(keys).any()
n_before = len(panel_all)
panel_all = panel_all.merge(prop_cy, on=keys, how="left", indicator=True)
assert len(panel_all) == n_before
print("panel rows with property data:", (panel_all["_merge"] == "both").sum(), "of", n_before,
      "| property-only country-years not added:", len(prop_cy) - (panel_all["_merge"] == "both").sum())
panel_all = panel_all.drop(columns="_merge")
panel_all.groupby("year")[["burglary_rate", "theft_rate", "property_rate"]].count().tail(8)

panel rows with property data: 2130 of 3332 | property-only country-years not added: 3


,burglary_rate,theft_rate,property_rate
year,,,
2017,81,85,77
2018,88,92,84
2019,79,87,76
2020,83,92,80
2021,74,84,73
2022,79,90,79
2023,60,69,60
2024,61,72,61


### 4b. Adding World Bank socioeconomic indicators
The World Bank file (described in section 1e) is pivoted to one row per country-year and left-joined to the panel on ISO code and year.

GDP per capita (PPP) and the Gini index describe income and inequality. Total population is used to check the population implied by UNODC's counts and rates.

Details to be aware of: Gini is only measured in some years for some countries; UNODC reports the three UK nations separately, so each gets the UK-wide World Bank value; Iraq's sub-entries, several overseas territories (e.g. Guadeloupe, Martinique, Réunion) and the Holy See have no World Bank match.

In [19]:
print(wb_raw.shape, "| indicators:", wb_raw["indicator"].unique().tolist())
print("share of country-years with a value:")
print(wb_raw.groupby("indicator")["value"].apply(lambda x: x.notna().mean()).round(2).to_string())

wb_wide = wb_raw.pivot_table(index=["iso3", "year"], columns="indicator", values="value").reset_index()
socio_cols = ["gdp_pc_ppp", "gini"]

# UNODC splits the UK and Iraq into sub-entries; map UK nations to the UK-wide World Bank code
iso_map = {"GBR_E_W": "GBR", "GBR_S": "GBR", "GBR_NI": "GBR"}
panel_all["wb_iso3"] = panel_all["iso3"].replace(iso_map)
panel_all = panel_all.merge(wb_wide.rename(columns={"iso3": "wb_iso3", "population": "wb_population"}), on=["wb_iso3", "year"], how="left")

unmatched = sorted(set(panel_all.loc[panel_all["wb_population"].isna(), "country"]))
print("\njurisdictions with no World Bank match:", unmatched)
print("rows with GDP:", f"{panel_all.gdp_pc_ppp.notna().mean():.0%}", "| with Gini:", f"{panel_all.gini.notna().mean():.0%}")

(19950, 6) | indicators: ['gdp_pc_ppp', 'gini', 'population']
share of country-years with a value:
indicator
gdp_pc_ppp    0.92
gini          0.28
population    1.00

jurisdictions with no World Bank match: ['Anguilla', 'Cook Islands', 'French Guiana', 'Guadeloupe', 'Holy See', 'Iraq (Central Iraq)', 'Iraq (Kurdistan Region)', 'Martinique', 'Mayotte', 'Montserrat', 'Réunion', 'Saint Helena', 'Saint Pierre and Miquelon']
rows with GDP: 93% | with Gini: 46%


In [20]:
# Validation: the population we inferred from UNODC counts/rates should be close to the World Bank population
chk = panel_all.dropna(subset=["pop", "wb_population"]).query("not territory_or_subnational")
ratio = (chk["pop"] / chk["wb_population"])
print("UNODC-implied / World Bank population: median %.3f, 5th-95th pct %.2f-%.2f (n=%d)" % (ratio.median(), ratio.quantile(.05), ratio.quantile(.95), len(chk)))

UNODC-implied / World Bank population: median 1.000, 5th-95th pct 0.57-1.08 (n=2695)


## 5. Coverage: which countries and years can we compare?
**Analysis window: 2010–2024.** Our group agreed that all three parts of the project (Seattle, United States, world) will use the years 2010–2024, so that the data at the three scales cover the same period and can be compared directly. The UNODC files run 2003–2024, so they cover the whole window; earlier years stay in the source tables but are not analyzed. This section checks how many countries have enough data inside the window, and how complete the most recent years are.

In [21]:
cov = panel_all.groupby("year")[rate_cols].count()
cov.columns = ["homicide", "assault", "robbery", "rape", "kidnapping"]
cov_long = cov.reset_index().melt("year", var_name="offense", value_name="jurisdictions")
alt.Chart(cov_long, title="Jurisdictions with a reported rate, by year and offense").mark_line(point=True).encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("jurisdictions:Q", title="Jurisdictions"),
    color=alt.Color("offense:N", title="Offense", sort=list(cov.columns)),
    tooltip=["year", "offense", "jurisdictions"],
).properties(width=650, height=280)

alt.Chart(...)

In [22]:
# How many jurisdictions report in the agreed window, at different completeness thresholds?
W_START, W_END = 2010, 2024
n_years = W_END - W_START + 1
w = panel_all[panel_all.year.between(W_START, W_END)].groupby("country")[["hom_rate", "assault_rate", "robbery_rate", "property_rate"]].count()
thr = {"at least 1 year": 1, "at least 10 of 15 years": 10, "at least 12 of 15 years": 12, "all 15 years": n_years}
display(pd.DataFrame({k: (w >= v).sum() for k, v in thr.items()}).rename(index=lambda c: c.replace("_rate", "")).rename_axis(f"jurisdictions, {W_START}-{W_END}"))
# the last two years are still incomplete: jurisdictions reporting a homicide rate in each recent year
panel_all[panel_all.year >= 2019].groupby("year")["hom_rate"].count().rename("jurisdictions with a homicide rate").to_frame().T

,at least 1 year,at least 10 of 15 years,at least 12 of 15 years,all 15 years
"jurisdictions, 2010-2024",,,,
hom,192,123,110,70
assault,143,94,78,50
robbery,138,81,69,39
property,119,75,54,33


year,2019,2020,2021,2022,2023,2024
jurisdictions with a homicide rate,140,136,126,119,108,95


In [23]:
# How many of the 15 years in the window does each jurisdiction report a homicide rate for?
yrs = (panel_all[panel_all.year.between(W_START, W_END)].groupby(["country", "region"])["hom_rate"].count()
       .rename("years").reset_index().query("years > 0"))
alt.Chart(yrs, title=alt.Title(f"Homicide data: how many years does each jurisdiction report, {W_START}-{W_END}?",
                               subtitle="Solid bars: 10 or more years, kept for the analysis. Faded bars: too few years, set aside.")).mark_bar().encode(
    x=alt.X("years:O", title="Years with a reported homicide rate (out of 15)", scale=alt.Scale(domain=list(range(1, n_years + 1))),
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Jurisdictions"),
    color=alt.Color("region:N", title="Region"),
    opacity=alt.condition(alt.datum.years >= 10, alt.value(1), alt.value(0.35)),
    tooltip=["years", "region", alt.Tooltip("count():Q", title="jurisdictions")],
).properties(width=650, height=280)

alt.Chart(...)

In [24]:
# Analysis window agreed by the group: 2010-2024,
# countries with a homicide rate in >= 10 of those 15 years; sovereign countries only for rankings.
START, END, MIN_YEARS = 2010, 2024, 10
RECENT = (END - 4, END)          # 'recent level' = mean of the last five years of the window
nyr = panel_all[panel_all.year.between(START, END)].groupby("country")["hom_rate"].count()
keep = nyr[nyr >= MIN_YEARS].index
panel = panel_all[panel_all.country.isin(keep) & panel_all.year.between(START, END)].copy()
print(f"{len(keep)} jurisdictions kept for {START}-{END}")
print(panel.groupby("region")["country"].nunique().sort_values(ascending=False).to_string())
# panel_all keeps everything; nothing is discarded from the source files.

123 jurisdictions kept for 2010-2024
region
Europe      42
Americas    37
Asia        29
Africa      11
Oceania      4


## 6. Exploratory look: the United States among its peer countries

The group's theme runs from Seattle to the United States to the world, so the most useful world-scale comparison is the United States against a small set of comparable countries, not against all 126.

**Peer countries (8):** Canada, France, Germany, Italy and Japan (the other members of the G7), plus Australia, Spain and the Netherlands. All are large, high-income democracies with at least 14 years of homicide data in 2010–2024. The United Kingdom would belong in this list but is left out because UNODC reports its three nations separately and made this analysis even more complex. Crime was not used to choose the peers.

In [25]:
ANCHOR = "United States of America"
# Peer countries: the other G7 members (without the UK, which UNODC reports as three nations) plus Australia, Spain and the Netherlands
peers = ["Canada", "France", "Germany", "Italy", "Japan", "Australia", "Spain", "Netherlands (Kingdom of the)"]
assert set(peers) <= set(panel.country), "a peer country is missing from the panel"
cand = panel[~panel.territory_or_subnational & ~panel.small_pop]     # sovereign countries with at least 1M people (context in the scatterplots)
SHORT = {"United States of America": "United States", "Russian Federation": "Russia", "Republic of Korea": "South Korea",
         "Netherlands (Kingdom of the)": "Netherlands"}
US, PEER, OTHER = "United States", "Peer countries", "Other countries"
US_COLOR, PEER_COLOR, OTHER_COLOR = "#d62728", "#4c72b0", "#c8c8c8"   # red = United States, blue = peers, gray = all other countries

def group_color(groups, legend=True):
    """One color convention for every chart."""
    pal = {US: US_COLOR, PEER: PEER_COLOR, OTHER: OTHER_COLOR}
    return alt.Color("group:N", title=None, scale=alt.Scale(domain=groups, range=[pal[g] for g in groups]), legend=alt.Legend() if legend else None)

def us_axis(**kw):
    """Axis whose 'United States' label is red and bold."""
    is_us = "datum.value == 'United States'"
    return alt.Axis(labelColor=alt.condition(is_us, alt.value(US_COLOR), alt.value("black")),
                    labelFontWeight=alt.condition(is_us, alt.value("bold"), alt.value("normal")), **kw)

panel[panel.country.isin([ANCHOR] + peers)].groupby("country")["hom_rate"].count().rename("years with a homicide rate").to_frame()

,years with a homicide rate
country,
Australia,15
Canada,15
France,15
Germany,15
Italy,15
Japan,15
Netherlands (Kingdom of the),15
Spain,15
United States of America,14


Viz 4 mock-up (how alike are the US and its peers?): average income and inequality, 2010–2024, US highlighted.

In [26]:
prof = panel[panel.country.isin([ANCHOR] + peers)].groupby("country")[socio_cols].mean().rename(index=SHORT)

def profile_bars(col, title, fmt):
    d = prof[col].dropna().rename("value").rename_axis("country").reset_index()
    d["group"] = np.where(d.country == US, US, PEER)
    return alt.Chart(d, title=title).mark_bar().encode(
        x=alt.X("value:Q", title=None, axis=alt.Axis(format=fmt)),
        y=alt.Y("country:N", sort="-x", title=None, axis=us_axis()),
        color=group_color([US, PEER]),
        tooltip=["country", alt.Tooltip("value:Q", format=fmt)],
    ).properties(width=300, height=220)

display(alt.hconcat(profile_bars("gdp_pc_ppp", "GDP per capita, PPP (international $)", ",.0f"),
                    profile_bars("gini", "Gini index (higher = more unequal)", ".1f"),
                    title=alt.Title(f"Income and inequality in the United States and its peer countries, mean {START}-{END}", anchor="middle", fontSize=15)))
prof.round(1).sort_values("gdp_pc_ppp", ascending=False)

alt.HConcatChart(...)

,gdp_pc_ppp,gini
country,,
United States,61307.6,41.2
Netherlands,59737.6,27.8
Germany,55215.7,31.7
Australia,52442.3,34.2
Canada,50287.1,32.6
France,46608.0,32.1
Italy,44346.1,34.9
Japan,43219.8,32.0
Spain,40571.8,35.0


Viz 1 mock-up (is the US high or low among its peers?): ranked dot plots, mean of 2020–2024 (the years available), US highlighted. Homicide is the comparable measure; robbery and burglary are shown with the usual warning that definitions and reporting differ by country.

In [27]:
grp = panel[panel.country.isin([ANCHOR] + peers) & panel.year.between(*RECENT)]
recent = grp.groupby("country")[["hom_rate", "robbery_rate", "burglary_rate"]].mean().rename(index=SHORT)

def ranked_dots(col, title):
    d = recent[col].dropna().rename("rate").rename_axis("country").reset_index()
    d["group"] = np.where(d.country == US, US, PEER)
    return alt.Chart(d, title=f"{title}, n = {len(d)}").mark_point(filled=True, size=120, opacity=1).encode(
        x=alt.X("rate:Q", scale=alt.Scale(type="log"), axis=alt.Axis(values=[0.1, 1, 10, 100, 1000], format="~g"),
                title=f"Rate per 100,000 (log scale), mean {RECENT[0]}-{RECENT[1]}"),
        y=alt.Y("country:N", sort="-x", title=None, axis=us_axis(grid=True)),
        color=group_color([US, PEER]),
        tooltip=["country", alt.Tooltip("rate:Q", format=".1f")],
    ).properties(width=220, height=260)

display(alt.hconcat(ranked_dots("hom_rate", "Homicide victims"), ranked_dots("robbery_rate", "Robbery (recorded)"),
                    ranked_dots("burglary_rate", "Burglary (recorded)"),
                    title=alt.Title("The United States among its peer countries", anchor="middle", fontSize=15)))
recent.round(1).sort_values("hom_rate", ascending=False)

alt.HConcatChart(...)

,hom_rate,robbery_rate,burglary_rate
country,,,
United States,6.4,72.9,267.6
Canada,2.1,55.4,334.3
France,1.2,100.4,440.8
Germany,0.9,45.2,332.3
Australia,0.8,37.8,560.4
Netherlands,0.7,34.8,215.6
Spain,0.7,121.0,294.4
Italy,0.5,42.7,225.3
Japan,0.2,1.0,32.8


Viz 2 mock-up (is the US following the same trend as its peers?): homicide rate over time, peers in blue and the US in red.

In [28]:
tl = panel[panel.country.isin([ANCHOR] + peers)].pivot_table(index="year", columns="country", values="hom_rate").rename(columns=SHORT)
tl_long = tl.reset_index().melt("year", var_name="country", value_name="rate").dropna()
tl_long["group"] = np.where(tl_long.country == US, US, PEER)
ends = tl_long.sort_values("year").groupby("country").tail(1).sort_values("rate").copy()   # last point of each line, for the direct labels
label_y = []                                                   # push labels apart so they do not overlap on the log axis
for r in ends.rate:
    label_y.append(r if not label_y else max(r, label_y[-1] * 1.26))
ends["label_y"] = label_y
x_year = alt.X("year:Q", title=None, axis=alt.Axis(format="d", values=list(range(START, END + 1, 2))), scale=alt.Scale(domain=[START, END + 2.5]))
y_rate = alt.Y("rate:Q", title="Homicide victims per 100,000 (log scale)", scale=alt.Scale(type="log"), axis=alt.Axis(values=[0.2, 0.5, 1, 2, 5, 10, 20], format="~g"))

def trend_layer(group, width, opacity):
    d, e = tl_long[tl_long.group == group], ends[ends.group == group]
    line = alt.Chart(d).mark_line(strokeWidth=width, opacity=opacity).encode(
        x=x_year, y=y_rate, detail="country:N", color=group_color([US, PEER], legend=False),
        tooltip=["country", "year", alt.Tooltip("rate:Q", format=".1f")])
    label = alt.Chart(e).mark_text(align="left", dx=6, fontSize=11 if group == US else 9, fontWeight="bold" if group == US else "normal").encode(
        x=x_year, y=alt.Y("label_y:Q", scale=alt.Scale(type="log")), text="country:N", color=group_color([US, PEER], legend=False))
    return line + label

display((trend_layer(PEER, 1.3, .75) + trend_layer(US, 3.5, 1)).properties(
    width=640, height=360, title="Homicide rate, United States (red) and its peer countries (blue)"))
# change between the first three and the last three years of the window (averaging the years available, since 2023-24 are incomplete)
first3, last3 = tl.loc[START:START + 2].mean(), tl.loc[END - 2:END].mean()
chg_p = (last3 / first3 - 1).mul(100).sort_values().round(0)
print(f"Change in homicide rate, mean {START}-{START + 2} to mean {END - 2}-{END} (%):"); print(chg_p.to_string())
print("\nyears with data in", f"{END - 2}-{END}:"); print(tl.loc[END - 2:END].notna().sum().to_string())

alt.LayerChart(...)

Change in homicide rate, mean 2010-2012 to mean 2022-2024 (%):
country
Italy           -37.0
Japan           -34.0
Australia       -18.0
Netherlands     -16.0
Spain           -14.0
Germany          -2.0
France            2.0
Canada           27.0
United States    31.0

years with data in 2022-2024:
country
Australia        3
Canada           3
France           3
Germany          3
Italy            3
Japan            3
Netherlands      3
Spain            3
United States    2


Viz 3 mock-up (does crime track national economic conditions, and where does the US sit?): all countries in light gray, peers in blue, the US in red.

In [29]:
avg2 = cand.groupby(["country", "region"])[["hom_rate"] + socio_cols].mean().reset_index()      # country averages over the window

def peer_scatter(sc, lab, log_x):
    d = avg2.dropna(subset=[sc, "hom_rate"]); d = d[d.hom_rate > 0].copy()
    d["group"] = np.select([d.country == ANCHOR, d.country.isin(peers)], [US, PEER], OTHER)
    d["name"] = d.country.replace(SHORT)
    d = d.sort_values("group", key=lambda g: g.map({OTHER: 0, PEER: 1, US: 2}))        # draw the US last, on top
    x = alt.X(f"{sc}:Q", title=lab, scale=alt.Scale(type="log" if log_x else "linear", zero=False, nice=False),
              axis=alt.Axis(values=[2000, 5000, 10000, 20000, 50000, 100000], format=",.0f") if log_x else alt.Axis())
    y = alt.Y("hom_rate:Q", title="Homicide rate per 100,000 (log scale)", scale=alt.Scale(type="log"), axis=alt.Axis(values=[0.2, 0.5, 1, 2, 5, 10, 20, 50], format="~g"))
    pts = alt.Chart(d).mark_point(filled=True, opacity=1).encode(
        x=x, y=y, color=group_color([US, PEER, OTHER]),
        size=alt.Size("group:N", scale=alt.Scale(domain=[US, PEER, OTHER], range=[160, 70, 30]), legend=None),
        tooltip=["name", alt.Tooltip(f"{sc}:Q", format=",.1f"), alt.Tooltip("hom_rate:Q", format=".1f")])
    lab_peer = alt.Chart(d[d.group == PEER]).mark_text(align="left", dx=6, dy=-5, fontSize=9, color=PEER_COLOR).encode(x=x, y=y, text="name:N")
    lab_us = alt.Chart(d[d.group == US]).mark_text(align="left", dx=8, dy=-8, fontSize=12, fontWeight="bold", color=US_COLOR).encode(x=x, y=y, text="name:N")
    return (pts + lab_peer + lab_us).properties(width=360, height=320, title=f"n = {len(d)} countries")

alt.hconcat(peer_scatter("gdp_pc_ppp", "GDP per capita, PPP (log scale)", True), peer_scatter("gini", "Gini index (higher = more unequal)", False),
            title=alt.Title(f"Homicide vs income and inequality, country averages {START}-{END}", anchor="middle", fontSize=15))

alt.HConcatChart(...)

Viz 5 mock-up (crime mix: is the US unusually property-crime heavy?): recorded property crime (burglary + theft) against recorded violent crime (homicide + serious assault + robbery + rape), mean of 2020–2024 (the years available), for the US and the peers that report all of them. Left: the mix. Right: the rates behind it. Both depend heavily on each country's definitions and reporting, so this supports broad contrasts only.

In [30]:
viol_parts, prop_parts = ["hom_rate", "assault_rate", "robbery_rate", "rape_rate"], ["burglary_rate", "theft_rate"]
gm = grp.groupby("country")[viol_parts + prop_parts].mean().dropna().rename(index=SHORT)
mix = pd.DataFrame({"Burglary": gm.burglary_rate, "Theft": gm.theft_rate, "Violent (4 offenses)": gm[viol_parts].sum(axis=1)})
prop_share = (mix.Burglary + mix.Theft) / mix.sum(axis=1)
order = prop_share.sort_values(ascending=False).index.tolist()            # most property-heavy first
tot = pd.DataFrame({"Property": mix.Burglary + mix.Theft, "Violent": mix["Violent (4 offenses)"]})
y_mix = alt.Y("country:N", sort=order, title=None, axis=us_axis())

mix_long = mix.rename_axis("country").reset_index().melt("country", var_name="offense", value_name="rate")
mix_long["part"] = mix_long.offense.map({"Burglary": 0, "Theft": 1, "Violent (4 offenses)": 2})
left = alt.Chart(mix_long, title=f"Mix of recorded crime, mean {RECENT[0]}-{RECENT[1]} (n = {len(mix)})").mark_bar().encode(
    y=y_mix, x=alt.X("rate:Q", stack="normalize", title="Share of recorded property + violent offenses", axis=alt.Axis(format="%")),
    color=alt.Color("offense:N", title=None, scale=alt.Scale(domain=list(mix.columns), range=["#9ecae1", "#4c72b0", "#c44e52"]), legend=alt.Legend(orient="bottom")),
    order="part:Q", tooltip=["country", "offense", alt.Tooltip("rate:Q", format=",.0f", title="per 100,000")],
).properties(width=380, height=240)

tot_long = tot.rename_axis("country").reset_index().melt("country", var_name="kind", value_name="rate")
right = alt.Chart(tot_long, title="Rates behind the mix").mark_bar().encode(
    y=y_mix, yOffset=alt.YOffset("kind:N", sort=["Property", "Violent"]), x=alt.X("rate:Q", title="Recorded offenses per 100,000"),
    color=alt.Color("kind:N", title=None, scale=alt.Scale(domain=["Property", "Violent"], range=["#4c72b0", "#c44e52"]), legend=alt.Legend(orient="bottom")),
    tooltip=["country", "kind", alt.Tooltip("rate:Q", format=",.0f")],
).properties(width=300, height=240)

display(alt.hconcat(left, right).resolve_scale(color="independent"))
pd.concat([tot.round(0), prop_share.mul(100).round(0).rename("property share %")], axis=1).loc[order]

alt.HConcatChart(...)

,Property,Violent,property share %
country,,,
Netherlands,1449.0,76.0,95.0
Japan,351.0,19.0,95.0
Germany,1552.0,229.0,87.0
United States,1678.0,395.0,81.0
Spain,669.0,177.0,79.0
France,2344.0,731.0,76.0


## 7. Summary

Three UNODC files (homicide, violent and sexual crime, property crime), merged with World Bank indicators, into a country-year panel (210 jurisdictions, 2003–2024). The analysis uses 2010–2024, the window the group agreed on so that the Seattle, United States and world data match. In that window 123 jurisdictions have at least 10 years of homicide data (42 Europe, 37 Americas, 29 Asia, 11 Africa, 4 Oceania). Homicide is by far the best-covered measure (94 jurisdictions for assault, 81 for robbery, 75 for property crime at the same threshold). The last two years are still incomplete: 108 jurisdictions report a homicide rate for 2023 and 95 for 2024, against 140 for 2019. Coverage is skewed toward Europe and the Americas. Nothing was discarded from the source data.

**What the exploration shows (not final)**
- The peers to the US are eight large high-income countries: Canada, France, Germany, Italy, Japan, Australia, Spain and the Netherlands. The US has the highest income of the nine (GDP per capita about 61,000 international dollars, against 41,000 to 60,000) and is the most unequal (Gini 41, against 28 to 35). Its homicide rate for 2020–2024 (6.4 per 100,000) is the highest of the nine, about 3 to 30 times the rates of its peers (Canada 2.1, France 1.2, Germany 0.9, Australia 0.8, the Netherlands and Spain 0.7, Italy 0.5, Japan 0.2). Comparing 2010–12 with 2022–24, homicide fell in five of the eight peers, was flat in Germany and France, and rose in Canada (+27%) and the US (+31%). The UK is not in the list because UNODC reports its three nations separately.
- Among the six of these countries that report every category, recorded property crime (burglary + theft) is 81% of recorded property-plus-violent crime in the US for 2020–2024, in the middle of the range (95% in Japan and the Netherlands, 87% Germany, 79% Spain, 76% France). US burglary (268 per 100,000) is in the lower half of the nine. So on recorded data the US is not unusually property-crime heavy; it stands out on homicide. Recorded property and assault figures depend heavily on definitions and reporting, so this is a broad contrast only.
- Across countries, homicide is lower where income is higher (GDP per capita, 92 countries) and higher where inequality is greater (Gini index, 84 countries). The United States sits above the pattern for its income. These are patterns across country averages, not causal effects.

Here are five plots that we are interested in creating: Five graphics that compare the United States with its peer countries (a ranked dot plot, a peers-in-gray line chart, scatterplots against income and inequality, bar charts of the peers' income and inequality, and a property-versus-violent crime mix). They are specified in section 8 and mocked up in section 6.

## 8. Step 2: Visualization brainstorm
Five visualizations built around one question: how does the United States compare with other large high-income countries? All use the agreed 2010–2024 window and one color convention: red for the United States, blue for its peers, gray for all other countries. Each has a mock-up in section 6; in the final project they could sit together as one dashboard. A few further ideas are listed briefly at the end.

### Viz 1 — Is the US high or low among its peers?  *(mock-up: section 6)*
**Statement:** Among nine large high-income countries, the United States has the highest homicide rate, three or more times that of every peer.
- **Variables:** `country`, mean `hom_rate` (and `robbery_rate`, `burglary_rate`) for 2020–2024; filter to the US and its peers. New variable: a US / peer flag.
- **Encodings:** y = country (ranked), x = rate per 100,000, color = US vs peer.
- **Chart type:** ranked dot plot.
- **Design:** US in red and bold, peers blue, log axis because rates range from 0.2 to 6.4 per 100,000; one highlight color used consistently across all five charts.

### Viz 2 — Is the US following the same trend as its peers?  *(mock-up: section 6)*
**Statement:** Since 2010 homicide has fallen or stayed flat in seven of the eight peer countries but risen by about 30% in the United States.
- **Variables:** `year`, `hom_rate`, `country` for the US and peers, 2010–2024. Optionally index to 2010 = 100.
- **Encodings:** x = year, y = rate, one line per country, color = US vs peer.
- **Chart type:** multiple-line chart, peers in thin blue lines and the US in a thick red line.
- **Design:** direct labels at line ends instead of a legend, log y-axis (or indexed values) so low-rate peers stay readable, mark 2020.

### Viz 3 — Does crime track economic conditions, and where does the US sit?  *(mock-up: section 6)*
**Statement:** Richer and more equal countries have lower homicide rates, and the US has more homicide than its income predicts.
- **Variables:** country-average `gdp_pc_ppp`, `gini`, `hom_rate` (2010–2024); flag for peer and US.
- **Encodings:** x = income or inequality, y = homicide rate (log), color = other / peer / US.
- **Chart type:** faceted scatterplots.
- **Design:** all countries light gray for context, peers blue with labels, US red; show the number of countries in each panel title.

### Viz 4 — How alike are the US and its peers?  *(mock-up: section 6)*
**Statement:** The United States is the richest of the nine countries and also the most unequal.
- **Variables:** country-average `gdp_pc_ppp` and `gini` (2010–2024) for the US and its eight peers.
- **Encodings:** y = country (ranked), x = value, color = US vs peer; one panel per measure.
- **Chart type:** two ranked horizontal bar charts side by side.
- **Design:** US in red, peers blue, bars sorted so the US position is read at a glance; sets up Viz 3 by showing the peers differ from the US more in inequality than in income.

### Viz 5 — Crime mix: property vs violent  *(mock-up: section 6)*
**Statement:** The United States is not unusually property-crime heavy: about four in five recorded offenses are property crimes, in the middle of its peers' range.
- **Variables:** `burglary_rate`, `theft_rate` (property) and `hom_rate`, `assault_rate`, `robbery_rate`, `rape_rate` (violent), mean 2020–2024. Compute each group's share of the total; keep only countries reporting every category. Vehicle theft is inside theft and is not added.
- **Encodings:** y = country, x = share of recorded crime, color = offense group.
- **Chart type:** 100% stacked horizontal bar (grouped bars if absolute rates matter more than mix).
- **Design:** fixed segment order, US highlighted, strong caveat that recorded crime mixes reflect definitions and reporting; possibly restrict to peers with similar recording practices.

### Other ideas we are also considering
Listed only; not developed for this assignment.
- **A. Regional homicide trends:** line chart of world regions indexed to 2010 = 100.
- **B. Country ranking and consistency:** dot plot with min–max range bars for the highest and lowest countries.
- **C. Firearms and homicide:** scatterplot of firearm share against homicide rate.
- **D. Do violent crime measures agree?** small-multiple scatterplots of homicide against assault, robbery and rape.
- **E. Where did homicide rise or fall?** country × year heat map of change since 2010.
- **F. World map of the latest homicide rate:** choropleth.
- **G. Seattle against national and international trends:** indexed lines, once the group's data are combined.
- **H. Who is killed, and by whom?** stacked bar of victims by sex and relationship to the perpetrator.
- **I. National characteristics and violence:** correlation heat map against income and inequality.

## 9. References

**Data sets**

United Nations Office on Drugs and Crime. (2026a). *UNODC Research – Data Portal: Violent Crime & Sexual Violence*. https://data.unodc.org/datareport/violent-offences  (file: `data/data_cts_violent_and_sexual_crime.xlsx`)

United Nations Office on Drugs and Crime. (2026b). *UNODC Research – Data Portal: Intentional Homicide*. https://data.unodc.org/datareport/hom-victim  (file: `data/data_cts_intentional_homicide.xlsx`, sheet `data_cts_intentional_homicide`)

United Nations Office on Drugs and Crime. (2026c). *UNODC Research – Data Portal: Corruption and Economic Crime*. https://data.unodc.org/datareport/econ-corruption  (file: `data/data_cts_corruption_and_economic_crime.xlsx`)

World Bank. (2026). *World Development Indicators*. Retrieved November 14, 2026, from https://api.worldbank.org  (file: `data/worldbank_indicators_2000_2024.csv`, built by `code/fetch_worldbank.py`)
